<a href="https://colab.research.google.com/github/Sathvikabanavath/GenAI-Tasks/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.4 MB/s eta 0:00:00


In [ ]:
documents = [
    "The capital of France is Paris. It is known for the Eiffel Tower.",
    "Python is a high-level programming language created by Guido van Rossum.",
    "The Great Wall of China is over 13,000 miles long.",
    "The Earth revolves around the Sun and takes about 365 days to complete one orbit.",
    "Albert Einstein developed the theory of relativity.",
    "Water boils at 100 degrees Celsius at sea level.",
    "The human heart has four chambers.",
    "Mount Everest is the highest mountain in the world.",
]

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
embeddings = embedder.encode(documents, convert_to_numpy=True)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def retrieve_chunks(query, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(query_embedding, k)
    return [documents[i] for i in indices[0]]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def rag_answer(question, k=3):
    # Retrieve context
    contexts = retrieve_chunks(question, k)
    context_text = " ".join(contexts)

    # Prompt
    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context_text}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
questions = [
    "What is the capital of France?",
    "Who created Python?",
    "How long does the Earth take to orbit the Sun?",
    "Who developed the theory of relativity?",
    "What is the highest mountain in the world?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print("-" * 60)

Q: What is the capital of France?
A: Paris
------------------------------------------------------------
Q: Who created Python?
A: Guido van Rossum
------------------------------------------------------------
Q: How long does the Earth take to orbit the Sun?
A: 365 days
------------------------------------------------------------
Q: Who developed the theory of relativity?
A: Albert Einstein
------------------------------------------------------------
Q: What is the highest mountain in the world?
A: Mount Everest
------------------------------------------------------------
